# Workshop 1 — Preprocessing, Model Training & Serialization
**Module:** From Data to Deployment: Python, APIs & ML  
**Dataset:** Titanic (Kaggle)  
**Deliverable:** `../api/model.pkl` + this notebook

Pipeline: `train.csv → clean → encode → train DecisionTree → save model.pkl`

## Task 1 — Load & Preprocess

In [ ]:
import pandas as pd

df = pd.read_csv('../data/train.csv')
print(df.shape)
df.head()

In [ ]:
# Drop columns that are too noisy or mostly missing
df = df.drop(columns=['PassengerId', 'Name', 'Ticket', 'Cabin'])

# Impute missing Age with median, missing Embarked with mode
df['Age'] = df['Age'].fillna(df['Age'].median())
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

# Encode Sex: male=0, female=1
df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})

# Encode Embarked: C=0, Q=1, S=2
df['Embarked'] = df['Embarked'].map({'C': 0, 'Q': 1, 'S': 2})

# Verify: no nulls remain
print('Nulls remaining:', df.isnull().sum().sum())
df.head()

In [ ]:
from sklearn.model_selection import train_test_split

FEATURES = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']
TARGET   = 'Survived'

X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print('Train:', X_train.shape, '  Test:', X_test.shape)

## Task 2 — Train & Evaluate Decision Tree

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

model = DecisionTreeClassifier(max_depth=4, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print('Accuracy:', round(accuracy_score(y_test, y_pred), 4))
print('Confusion matrix:')
print(confusion_matrix(y_test, y_pred))

In [ ]:
# Optional: try different depths to see overfitting
for depth in [2, 3, 4, 5, None]:
    m = DecisionTreeClassifier(max_depth=depth, random_state=42)
    m.fit(X_train, y_train)
    acc = accuracy_score(y_test, m.predict(X_test))
    print(f'max_depth={depth}  →  test accuracy={acc:.4f}')

## Task 3 — Save model.pkl

In [ ]:
import pickle

with open('../api/model.pkl', 'wb') as f:
    pickle.dump(model, f)

print('Saved to ../api/model.pkl')

# Quick reload check
with open('../api/model.pkl', 'rb') as f:
    loaded = pickle.load(f)

print('Reload check — accuracy:', round(accuracy_score(y_test, loaded.predict(X_test)), 4))